In [65]:
import re
import unicodedata

FILE_TO_PROCESS = "nova-zemya.txt"

In [66]:
def clean_from_chitanka(text: str) -> str:
    # Strip start
    match = re.search(r'^\s*I\.\s+[\u0400-\u04FF]+', text, flags=re.MULTILINE)
    if match:
        text = text[match.start():]

    # Strip end
    text = re.sub(r'\$id.*', '', text, flags=re.DOTALL)

    # Strip part headers: "Част първа", "Част втора", etc.
    text = re.sub(r'^\s*Част\s+\w+\s*$', '', text, flags=re.MULTILINE)

    # Strip numbered chapter titles: "I. Бяла черква", "II. Нещо", etc.
    text = re.sub(r'^\s*[IVXLCDM]+\.\s+.{1,100}\s*$', '', text, flags=re.MULTILINE)

    # Strip references
    text = re.sub(r'\[.*?\]', '', text, flags = re.DOTALL)

    # Convert "’" to " "
    text = re.sub(r'’', ' ', text)

    # Convert cyrillic accented e to normal cyrillic e
    text = text.replace('\u0450', '\u0435')

    # Convert latin accented e to normal cyrillic e
    text = text.replace('\u00e8', '\u0435')

    # Convert latin accented o to normal cyrillic o
    text = text.replace('\u00f2', '\u043e')

    # Convert latin accented a to normal cyrillic a
    text = text.replace('\u00e0', 'u\0430')
    text = text.replace('\u00e1', 'u\0430')

    # Convert "Ѣ" to "я"
    text = text.replace('\u0462', '\u044f')

    # Remove words containing any latin characters
    text = re.sub(r'\b\w*[a-zA-Z]\w*\b', '', text)

    # Strip unneeded formatting
    text = re.sub(r'^\t\* \* \*\n', '', text, flags=re.MULTILINE)
    text = re.sub(r'^\t……+\n', '', text, flags=re.MULTILINE)
    text = re.sub(r'[\t_*`´«»]', '', text)

    # Convert digits to # (1893 -> ####)
    text = re.sub(r'\d', '#', text)

    # Normalize double quotation („“ -> ")
    text = re.sub(r'[„“]', '"', text)

    # Convert "№" to "номер"
    text = re.sub(r'№', 'номер', text)

    # Collapse 3+ newlines to 1 (preserve paragraph breaks)
    text = re.sub(r'\n{3,}', '\n', text)

    # Strip leading/trailing whitespace
    text = text.strip()

    return text

In [67]:
def print_corpus_chars(corpus):
    print(f"Characters: {len(corpus):,}")
    print(f"Unique chars: {len(set(corpus))}")
    print(f"Sample:\n{corpus[:300]}")

In [68]:
def print_non_ascii_letters(text):
    non_ascii_letters = set()

    for ch in text:
        if unicodedata.category(ch).startswith('L') and ord(ch) > 127:
            non_ascii_letters.add((repr(ch), unicodedata.name(ch, '?')))

    for char_repr, char_name in non_ascii_letters:
        print(char_repr, char_name)

In [69]:
def open_file():
    with open(f"../data/raw/{FILE_TO_PROCESS}", "r", encoding="utf-8") as f:
        return f.read()


In [70]:
def save_file(text):
    file_to_save = FILE_TO_PROCESS.split(".")[0] + "-processed.txt"
    with open(f"../data/processed/{file_to_save}", "w", encoding="utf-8") as f:
        f.write(text)

In [71]:
raw_txt = open_file()
print_non_ascii_letters(raw_txt)

'Б' CYRILLIC CAPITAL LETTER BE
'Л' CYRILLIC CAPITAL LETTER EL
'К' CYRILLIC CAPITAL LETTER KA
'П' CYRILLIC CAPITAL LETTER PE
'Д' CYRILLIC CAPITAL LETTER DE
'Е' CYRILLIC CAPITAL LETTER IE
'Ж' CYRILLIC CAPITAL LETTER ZHE
'З' CYRILLIC CAPITAL LETTER ZE
'Ш' CYRILLIC CAPITAL LETTER SHA
'Г' CYRILLIC CAPITAL LETTER GHE
'ù' LATIN SMALL LETTER U WITH GRAVE
'д' CYRILLIC SMALL LETTER DE
'ю' CYRILLIC SMALL LETTER YU
'б' CYRILLIC SMALL LETTER BE
'ц' CYRILLIC SMALL LETTER TSE
'й' CYRILLIC SMALL LETTER SHORT I
'х' CYRILLIC SMALL LETTER HA
'Х' CYRILLIC CAPITAL LETTER HA
'è' LATIN SMALL LETTER E WITH GRAVE
'à' LATIN SMALL LETTER A WITH GRAVE
'ô' LATIN SMALL LETTER O WITH CIRCUMFLEX
'е' CYRILLIC SMALL LETTER IE
'п' CYRILLIC SMALL LETTER PE
'о' CYRILLIC SMALL LETTER O
'ѐ' CYRILLIC SMALL LETTER IE WITH GRAVE
'Ч' CYRILLIC CAPITAL LETTER CHE
'т' CYRILLIC SMALL LETTER TE
'Ѣ' CYRILLIC CAPITAL LETTER YAT
'а' CYRILLIC SMALL LETTER A
'А' CYRILLIC CAPITAL LETTER A
'щ' CYRILLIC SMALL LETTER SHCHA
'Щ' CYRILLIC CAPIT

In [72]:
cleaned_txt = clean_from_chitanka(raw_txt)
save_file(cleaned_txt)
print_corpus_chars(cleaned_txt)

Characters: 756,426
Unique chars: 74
Sample:
По гладката, стръмна южна урва на Амбарица — високия старопланински връх, който гледа над Стремска долина, ставаше нещо необикновено и чудно: върволици свят пъплеше и се катереше нагоре въз върлото. Тия върволици идеха на синджир, защото вървяха по едничката козя пътека, що извиваше на безбройни лък
